# 衔接BND_convert.ipynb，绘制BND转换后的sv upset图。

## 一、需要用到的脚本，并存放在同一路径

### build_sv_upset_events_from_reclass_vcfs.py：

In [ ]:
#!/usr/bin/env python3
import argparse
import csv
import gzip
import re
from collections import defaultdict
from pathlib import Path


TOOLS = ["cue", "lumpy", "gridss", "manta", "delly", "svaba"]
BND_RE = re.compile(r"[\[\]]([^:\[\]]+):([0-9]+)[\[\]]")
GENERIC_TOKENS = {
    "", ".", "vcf", "vcfs", "result", "results", "output", "outputs", "variant", "variants",
    "work", "tmp", "temp", "filter", "filtered", "somatic", "somaticsv", "diploidsv",
    "candidatesv", "candidate", "final", "pon", "pon_filtered", "bnd_reclass",
    "cue", "lumpy", "gridss", "gripss", "manta", "delly", "svaba",
    "cue_result", "lumpy_result", "gridss_result", "manta_result", "delly_result", "svaba_result",
}


def open_text(path):
    if str(path).endswith(".gz"):
        return gzip.open(path, "rt")
    return open(path, "rt")


def parse_info(info):
    d = {}
    if info in {"", "."}:
        return d
    for item in info.split(";"):
        if not item:
            continue
        if "=" in item:
            k, v = item.split("=", 1)
            d[k] = v
        else:
            d[item] = True
    return d


def chrom_norm(chrom, keep_chr=True):
    chrom = str(chrom or "")
    if keep_chr:
        return chrom if chrom.lower().startswith("chr") else "chr" + chrom
    return chrom[3:] if chrom.lower().startswith("chr") else chrom


def chrom_key(chrom):
    c = chrom_norm(chrom, keep_chr=False)
    if c.isdigit():
        return (0, int(c))
    rank = {"X": 23, "Y": 24, "M": 25, "MT": 25}.get(c.upper())
    if rank is not None:
        return (0, rank)
    return (1, c)


def to_int(x, default=None):
    try:
        if x in {None, "", ".", "NA"}:
            return default
        return int(float(str(x).split(",")[0]))
    except Exception:
        return default


def norm_svtype(sv):
    sv = str(sv or "").strip().upper()
    if sv in {"", ".", "NA", "NAN", "NONE"}:
        return "UNKNOWN"
    if sv in {"DELETION"}:
        return "DEL"
    if sv in {"DUPLICATION"}:
        return "DUP"
    if sv in {"INVERSION"}:
        return "INV"
    if sv in {"INSERTION"}:
        return "INS"
    if sv in {"TRANSLOCATION", "CTX"}:
        return "TRA"
    if sv in {"BREAKEND"}:
        return "BND"
    return sv


def infer_tool(path):
    s = str(path).lower()
    for tool in TOOLS:
        if tool in s:
            return tool
    if "gripss" in s:
        return "gridss"
    return "unknown"


def strip_extensions(name):
    for suffix in [".vcf.gz", ".vcf"]:
        if name.endswith(suffix):
            return name[: -len(suffix)]
    return name


def clean_sample_token(name):
    x = strip_extensions(Path(name).name)
    patterns = [
        r"\.bnd_reclass$",
        r"_somatic$",
        r"\.somatic\.sv$",
        r"\.somatic\.indel$",
        r"\.somatic\.filtered$",
        r"\.somatic$",
        r"\.gripss\.pon_filtered$",
        r"\.gripss\.filtered$",
        r"\.gridss\.filtered$",
        r"\.gridss$",
        r"\.delly$",
        r"\.lumpy$",
        r"\.manta$",
        r"\.svaba$",
        r"_raw$",
    ]
    changed = True
    while changed:
        changed = False
        for pat in patterns:
            y = re.sub(pat, "", x, flags=re.IGNORECASE)
            if y != x:
                x = y
                changed = True
    for tool in TOOLS + ["gripss"]:
        x = re.sub(rf"(^|[._-]){tool}([._-]|$)", r"\1", x, flags=re.IGNORECASE)
    x = re.sub(r"[._-]+$", "", x)
    return x


def looks_like_sample_token(token):
    t = str(token or "").strip()
    if not t:
        return False
    if t.lower() in GENERIC_TOKENS:
        return False
    if not re.fullmatch(r"[A-Za-z]*[0-9]{3,}[A-Za-z]?", t):
        return False
    return True


def looks_like_pair_token(token):
    t = str(token or "").strip()
    if "_vs_" not in t:
        return False
    a, b = t.split("_vs_", 1)
    return looks_like_sample_token(a) and looks_like_sample_token(b)


def candidate_tokens(path):
    p = Path(path)
    raw = [p.name]
    for parent in list(p.parents)[:6]:
        raw.append(parent.name)

    tokens = []
    for item in raw:
        token = clean_sample_token(item)
        pair_matches = re.findall(
            r"([A-Za-z]*[0-9]{3,}[A-Za-z]?_vs_[A-Za-z]*[0-9]{3,}[A-Za-z]?)",
            token,
        )
        for pair_token in pair_matches:
            if looks_like_pair_token(pair_token):
                tokens.append(pair_token)
        if looks_like_sample_token(token):
            tokens.append(token)
            patient = patient_from_sample(token)
            if looks_like_sample_token(patient):
                tokens.append(patient)

    seen = set()
    out = []
    for token in tokens:
        if token not in seen:
            seen.add(token)
            out.append(token)
    return out


def patient_from_sample(sample):
    if sample.endswith(("T", "N", "P")) and len(sample) > 1:
        return sample[:-1]
    return sample


def read_pair_list(path):
    pairs = []
    with open(path, "rt") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        for row in reader:
            tumor = row.get("tumor_id") or row.get("tumor") or row.get("TUMOR") or row.get("sample_tumor")
            normal = row.get("normal_id") or row.get("normal") or row.get("NORMAL") or row.get("sample_normal")
            pair_id = row.get("pair_id") or row.get("pair")
            if not tumor or not normal:
                continue
            if not pair_id:
                pair_id = f"{tumor}_vs_{normal}"
            pairs.append({"pair_id": pair_id, "tumor_id": tumor, "normal_id": normal, "patient": patient_from_sample(tumor)})
    return pairs


def discover_pairs(files_by_tool):
    samples = set()
    patients_from_pair_files = set()
    for tool, paths in files_by_tool.items():
        for p in paths:
            for token in candidate_tokens(p):
                if looks_like_pair_token(token):
                    tumor, normal = token.split("_vs_", 1)
                    samples.add(tumor)
                    samples.add(normal)
                    patients_from_pair_files.add(patient_from_sample(tumor))
                elif token.endswith(("T", "N", "P")):
                    samples.add(token)
                else:
                    patients_from_pair_files.add(token)

    pairs = []
    by_patient = defaultdict(set)
    for sample in samples:
        by_patient[patient_from_sample(sample)].add(sample)

    for patient in sorted(set(by_patient) | patients_from_pair_files):
        ss = by_patient.get(patient, set())
        tumors = sorted([s for s in ss if s.endswith("T")])
        normals = sorted([s for s in ss if s.endswith(("N", "P"))])
        if tumors and normals:
            tumor = tumors[0]
            normal = normals[0]
        else:
            tumor = patient + "T"
            normal = patient + "N"
        pairs.append({"pair_id": f"{tumor}_vs_{normal}", "tumor_id": tumor, "normal_id": normal, "patient": patient})
    return pairs


def collect_vcfs(result_root):
    files_by_tool = {tool: [] for tool in TOOLS}
    for p in Path(result_root).rglob("*.vcf"):
        if ".ipynb_checkpoints" in p.parts:
            continue
        tool = infer_tool(p)
        if tool in files_by_tool:
            files_by_tool[tool].append(p)
    for p in Path(result_root).rglob("*.vcf.gz"):
        if ".ipynb_checkpoints" in p.parts:
            continue
        tool = infer_tool(p)
        if tool in files_by_tool:
            files_by_tool[tool].append(p)
    for tool in TOOLS:
        files_by_tool[tool] = sorted(set(files_by_tool[tool]))
    return files_by_tool


def index_vcfs(files_by_tool):
    idx = {tool: defaultdict(list) for tool in TOOLS}
    for tool, paths in files_by_tool.items():
        for p in paths:
            for token in candidate_tokens(p):
                idx[tool][token].append(p)
                idx[tool][patient_from_sample(token)].append(p)
    return idx


def choose_vcf(tool_index, tool, pair_id, tumor_id, normal_id, patient):
    candidates = []
    for key in [pair_id, tumor_id, patient, normal_id]:
        candidates.extend(tool_index.get(tool, {}).get(key, []))

    if not candidates:
        return "NA"

    pair_l = pair_id.lower()
    tumor_l = tumor_id.lower()
    normal_l = normal_id.lower()
    patient_l = patient.lower()

    def priority(path):
        s = str(path).lower()
        name = Path(path).name.lower()
        score = 1000

        # Best: explicit paired somatic result, e.g. 1866277T_vs_1866277N_somatic.bnd_reclass.vcf
        if pair_l in s:
            score -= 500
        if "somatic" in name:
            score -= 200

        # Never prefer raw single-sample files for paired somatic UpSet.
        if "_raw" in name or ".raw" in name:
            score += 1000

        # Prefer tumor-labelled file over normal-labelled file when pair-level name is absent.
        if tumor_l in name:
            score -= 80
        if normal_l in name:
            score += 80

        # Tool-specific preferences.
        if tool == "gridss":
            if "pon_filtered" in s:
                score -= 100
            if "pon_removed" in s:
                score += 500
            if "gripss" in s:
                score -= 20

        # Svaba paired result may be patient-level:
        # 1866277.svaba.somatic.sv.bnd_reclass.vcf
        if tool == "svaba" and patient_l in name and "somatic" in name:
            score -= 150

        if "bnd_reclass" in s:
            score -= 10
        if s.endswith(".vcf.gz"):
            score -= 2

        return (score, len(str(path)), str(path))

    candidates = sorted(set(candidates), key=priority)
    return str(candidates[0])


def alt_remote(alt):
    m = BND_RE.search(alt)
    if not m:
        return None, None
    return chrom_norm(m.group(1), keep_chr=True), int(m.group(2))


def count_vcf_filters(path):
    total = 0
    pass_records = 0
    nonpass_records = 0
    if path == "NA" or not Path(path).exists():
        return total, pass_records, nonpass_records
    with open_text(path) as handle:
        for line in handle:
            if line.startswith("#") or not line.strip():
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 8:
                continue
            total += 1
            filt = fields[6]
            if filt in {"PASS", "."}:
                pass_records += 1
            else:
                nonpass_records += 1
    return total, pass_records, nonpass_records


def parse_vcf_records(path, tool, pair, pass_only=True):
    if path == "NA" or not Path(path).exists():
        return []
    out = []
    with open_text(path) as handle:
        for line in handle:
            if line.startswith("#") or not line.strip():
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 8:
                continue
            chrom, pos, rec_id, ref, alt, qual, filt, info_s = fields[:8]
            if pass_only and filt not in {"PASS", "."}:
                continue
            info = parse_info(info_s)
            svtype = norm_svtype(info.get("SVTYPE"))
            pos1 = to_int(pos)
            if pos1 is None:
                continue

            chrom1 = chrom_norm(chrom, keep_chr=True)
            chrom2 = chrom_norm(info.get("CHR2") or info.get("CHROM2") or info.get("ENDCHR") or chrom1, keep_chr=True)
            pos2 = to_int(info.get("END") or info.get("POS2") or info.get("ENDPOS"))

            if svtype in {"BND", "TRA"}:
                a_chrom2, a_pos2 = alt_remote(alt)
                if a_chrom2:
                    chrom2 = a_chrom2
                if a_pos2:
                    pos2 = a_pos2

            if pos2 is None:
                svlen = to_int(info.get("SVLEN"))
                if svlen is not None and svlen != 0:
                    pos2 = pos1 + abs(svlen)
                else:
                    pos2 = pos1

            start = min(pos1, pos2) if chrom_norm(chrom1, False) == chrom_norm(chrom2, False) else pos1
            end = max(pos1, pos2) if chrom_norm(chrom1, False) == chrom_norm(chrom2, False) else pos2
            member_id = f"{tool}:{rec_id or chrom1 + ':' + str(pos1)}"
            out.append({
                "tool": tool,
                "pair_id": pair["pair_id"],
                "tumor_id": pair["tumor_id"],
                "normal_id": pair["normal_id"],
                "chrom1": chrom1,
                "pos1": pos1,
                "chrom2": chrom2,
                "pos2": pos2,
                "start": start,
                "end": end,
                "svtype": svtype,
                "member_id": member_id,
            })
    return out


def event_distance(a, b):
    same_chrom_pair = (
        chrom_norm(a["chrom1"], False) == chrom_norm(b["chrom1"], False)
        and chrom_norm(a["chrom2"], False) == chrom_norm(b["chrom2"], False)
    )
    swapped_chrom_pair = (
        chrom_norm(a["chrom1"], False) == chrom_norm(b["chrom2"], False)
        and chrom_norm(a["chrom2"], False) == chrom_norm(b["chrom1"], False)
    )
    if same_chrom_pair:
        return max(abs(a["pos1"] - b["pos1"]), abs(a["pos2"] - b["pos2"]))
    if swapped_chrom_pair:
        return max(abs(a["pos1"] - b["pos2"]), abs(a["pos2"] - b["pos1"]))
    return None


def compatible(a, b, max_dist, same_type_only):
    if same_type_only and a["svtype"] != b["svtype"]:
        return False
    dist = event_distance(a, b)
    return dist is not None and dist <= max_dist


def merge_events(records, max_dist, same_type_only):
    clusters = []
    for rec in sorted(records, key=lambda r: (chrom_key(r["chrom1"]), r["pos1"], chrom_key(r["chrom2"]), r["pos2"], r["svtype"])):
        hit = None
        for cluster in clusters:
            if any(compatible(rec, old, max_dist, same_type_only) for old in cluster):
                hit = cluster
                break
        if hit is None:
            clusters.append([rec])
        else:
            hit.append(rec)
    return clusters


def cluster_row(pair, idx, cluster):
    tools = sorted({r["tool"] for r in cluster}, key=TOOLS.index)
    svtypes = [r["svtype"] for r in cluster]
    svtype = max(set(svtypes), key=lambda x: (svtypes.count(x), -svtypes.index(x)))
    chrom1 = cluster[0]["chrom1"]
    chrom2 = cluster[0]["chrom2"]
    pos1 = int(round(sum(r["pos1"] for r in cluster) / len(cluster)))
    pos2 = int(round(sum(r["pos2"] for r in cluster) / len(cluster)))
    same = chrom_norm(chrom1, False) == chrom_norm(chrom2, False)
    start = min([r["start"] for r in cluster]) if same else pos1
    end = max([r["end"] for r in cluster]) if same else pos2
    row = {
        "cluster_id": f"SV{idx:06d}",
        "pair_id": pair["pair_id"],
        "tumor_id": pair["tumor_id"],
        "normal_id": pair["normal_id"],
        "chrom1": chrom1,
        "pos1": pos1,
        "chrom2": chrom2,
        "pos2": pos2,
        "start": start,
        "end": end,
        "svtype": svtype,
        "support_n": len(tools),
        "tools": ",".join(tools),
    }
    for tool in TOOLS:
        row[tool] = "1" if tool in tools else "0"
    row["member_record_count"] = len(cluster)
    row["member_record_ids"] = ",".join(r["member_id"] for r in cluster)
    return row


def write_manifest(path, pairs, tool_index):
    cols = ["pair_id", "tumor_id", "normal_id"] + [f"{tool}_vcf" for tool in TOOLS]
    with open(path, "wt") as out:
        out.write("\t".join(cols) + "\n")
        for pair in pairs:
            row = {
                "pair_id": pair["pair_id"],
                "tumor_id": pair["tumor_id"],
                "normal_id": pair["normal_id"],
            }
            for tool in TOOLS:
                row[f"{tool}_vcf"] = choose_vcf(tool_index, tool, pair["pair_id"], pair["tumor_id"], pair["normal_id"], pair["patient"])
            out.write("\t".join(str(row[c]) for c in cols) + "\n")


def main():
    p = argparse.ArgumentParser(description="Build six-tool SV UpSet manifest and merged event tables from BND-reclassified VCFs.")
    p.add_argument("--result-root", default="/mnt/home/ygjx/chenkejin/sv_tools_results_bnd_reclass")
    p.add_argument("--outdir", default="/mnt/home/ygjx/chenkejin/80_upset")
    p.add_argument("--pair-list", default=None, help="Optional TSV with tumor_id and normal_id columns.")
    p.add_argument("--max-distance", type=int, default=1000, help="Max breakpoint distance for merging events across tools.")
    p.add_argument("--same-type-only", action="store_true", help="Merge only records with the same SVTYPE.")
    p.add_argument("--include-filtered", action="store_true", help="Include non-PASS VCF records. Default: use PASS or . only.")
    args = p.parse_args()

    outdir = Path(args.outdir)
    events_dir = outdir / "events"
    events_dir.mkdir(parents=True, exist_ok=True)

    files_by_tool = collect_vcfs(args.result_root)
    tool_index = index_vcfs(files_by_tool)
    pairs = read_pair_list(args.pair_list) if args.pair_list else discover_pairs(files_by_tool)
    pairs = sorted(pairs, key=lambda x: x["pair_id"])

    manifest = outdir / "80_pairs.six_sv_tools.manifest.new.tsv"
    write_manifest(manifest, pairs, tool_index)

    event_cols = [
        "cluster_id", "pair_id", "tumor_id", "normal_id", "chrom1", "pos1", "chrom2", "pos2",
        "start", "end", "svtype", "support_n", "tools",
        "cue", "lumpy", "gridss", "manta", "delly", "svaba",
        "member_record_count", "member_record_ids",
    ]
    summary = outdir / "build_events.summary.tsv"
    audit = outdir / "build_events.vcf_filter_audit.tsv"
    with open(summary, "wt") as s, open(audit, "wt") as a:
        s.write("pair_id\ttumor_id\tnormal_id\traw_records\tmerged_events\toutput\n")
        a.write("pair_id\ttumor_id\tnormal_id\ttool\tvcf\ttotal_records\tpass_records\tnonpass_records\tused_records\tmode\n")
        for pair in pairs:
            records = []
            for tool in TOOLS:
                vcf = choose_vcf(tool_index, tool, pair["pair_id"], pair["tumor_id"], pair["normal_id"], pair["patient"])
                total_n, pass_n, nonpass_n = count_vcf_filters(vcf)
                tool_records = parse_vcf_records(vcf, tool, pair, pass_only=not args.include_filtered)
                records.extend(tool_records)
                mode = "ALL_RECORDS" if args.include_filtered else "PASS_ONLY"
                a.write(
                    f"{pair['pair_id']}\t{pair['tumor_id']}\t{pair['normal_id']}\t{tool}\t{vcf}\t"
                    f"{total_n}\t{pass_n}\t{nonpass_n}\t{len(tool_records)}\t{mode}\n"
                )
            clusters = merge_events(records, args.max_distance, args.same_type_only)
            rows = [cluster_row(pair, i, c) for i, c in enumerate(clusters, 1)]
            out_path = events_dir / f"{pair['pair_id']}.merged_sv_events.tsv"
            with open(out_path, "wt") as out:
                out.write("\t".join(event_cols) + "\n")
                for row in rows:
                    out.write("\t".join(str(row.get(c, "")) for c in event_cols) + "\n")
            s.write(f"{pair['pair_id']}\t{pair['tumor_id']}\t{pair['normal_id']}\t{len(records)}\t{len(rows)}\t{out_path}\n")

    print(f"[DONE] pairs={len(pairs)}")
    print(f"[MANIFEST] {manifest}")
    print(f"[EVENTS] {events_dir}")
    print(f"[SUMMARY] {summary}")
    print(f"[FILTER_AUDIT] {audit}")


if __name__ == "__main__":
    main()

### plot_upset_v7_svtype_size_from_events.py：

In [ ]:
#!/usr/bin/env python3
import argparse
import csv
import math
from collections import Counter, defaultdict
from pathlib import Path


TOOLS = ["cue", "lumpy", "gridss", "manta", "delly", "svaba"]
PLOT_TOOLS = ["gridss", "delly", "manta", "cue", "lumpy", "svaba"]
TOOL_LABELS = {
    "cue": "CUE",
    "lumpy": "LUMPY",
    "gridss": "GRIDSS",
    "manta": "MANTA",
    "delly": "DELLY",
    "svaba": "SVABA",
}
SVTYPE_PRIORITY = ["DEL", "DUP", "INV", "BND", "TRA", "INS", "SGL"]

# Edit colors here directly. The plotted colors will be exactly these values.
SVTYPE_COLORS = {
    "DEL": "#ec8e5a",
    "DUP": "#ec9e59",
    "INV": "#ecb66b",
    "BND": "#60aa84",
    "TRA": "#4098ac",
    "INS": "#53999d",
    "SGL": "#248d82",
    "UNKNOWN": "#9E9E9E",
}
SIZE_BINS = [
    ("50-100 bp", 50, 100),
    ("100 bp-1 kb", 100, 1_000),
    ("1-10 kb", 1_000, 10_000),
    ("10-100 kb", 10_000, 100_000),
    ("100 kb-1 Mb", 100_000, 1_000_000),
    (">1 Mb", 1_000_000, math.inf),
]


def parse_args():
    parser = argparse.ArgumentParser(
        description="Plot V7-style UpSet figures with SVTYPE stacked bars and aligned SV length track."
    )
    parser.add_argument(
        "--events-dir",
        default="/mnt/home/ygjx/chenkejin/80_upset/events",
        help="Directory containing *.merged_sv_events.tsv from the existing UpSet workflow.",
    )
    parser.add_argument(
        "--manifest",
        default="/mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv",
        help="Manifest used by the existing UpSet workflow. Required for --task-id.",
    )
    parser.add_argument(
        "--outdir",
        default="/mnt/home/ygjx/chenkejin/80_upset/upset_v7_svtype_size",
        help="Output directory.",
    )
    parser.add_argument("--pair-id", default=None, help="Run one pair id, e.g. 462745T_vs_462745N.")
    parser.add_argument("--task-id", type=int, default=None, help="1-based Slurm array task id in manifest.")
    parser.add_argument("--max-intersections", type=int, default=12, help="Number of top intersections to draw.")
    
    parser.add_argument("--png-dpi", type=int, default=220)
    return parser.parse_args()


def import_matplotlib():
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        return plt
    except Exception as exc:
        raise SystemExit(
            "Cannot import matplotlib. Install it in the active conda environment, for example:\n"
            "conda install -c conda-forge matplotlib\n"
            f"Original error: {exc}"
        )


def mkdirs(outdir):
    for sub in ["plots", "summary", "tables", "logs"]:
        (outdir / sub).mkdir(parents=True, exist_ok=True)


def read_manifest_pair(manifest, task_id):
    with open(manifest, "rt") as handle:
        header = handle.readline().rstrip("\n").split("\t")
        for line_no, line in enumerate(handle, start=1):
            if line_no != task_id:
                continue
            row = dict(zip(header, line.rstrip("\n").split("\t")))
            return row.get("pair_id") or f"{row.get('tumor_id')}_vs_{row.get('normal_id')}"
    raise SystemExit(f"No manifest row for task id {task_id}: {manifest}")


def event_paths(args):
    events_dir = Path(args.events_dir)
    if args.task_id is not None:
        pair_id = read_manifest_pair(args.manifest, args.task_id)
        return [events_dir / f"{pair_id}.merged_sv_events.tsv"]
    if args.pair_id:
        return [events_dir / f"{args.pair_id}.merged_sv_events.tsv"]
    return sorted(events_dir.glob("*.merged_sv_events.tsv"))


def read_events(path):
    rows = []
    with open(path, "rt") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        for row in reader:
            rows.append(row)
    return rows


def norm_svtype(value):
    sv = str(value or "").strip().upper()
    if sv in {"", ".", "NA", "NAN", "NONE"}:
        return "UNKNOWN"
    if sv in {"DEL", "DELETION"}:
        return "DEL"
    if sv in {"DUP", "DUPLICATION"}:
        return "DUP"
    if sv in {"INV", "INVERSION"}:
        return "INV"
    if sv in {"INS", "INSERTION"}:
        return "INS"
    if sv in {"TRA", "TRANSLOCATION", "CTX"}:
        return "TRA"
    if sv in {"BND", "BREAKEND"}:
        return "BND"
    if sv in {"SGL", "SINGLE_BREAKEND", "SINGLE"}:
        return "SGL"
    return sv


def ordered_svtypes(svtype_total):
    present = [sv for sv, count in svtype_total.items() if count > 0]
    priority = {sv: idx for idx, sv in enumerate(SVTYPE_PRIORITY)}
    return sorted(present, key=lambda sv: (priority.get(sv, len(SVTYPE_PRIORITY)), sv))


def svtype_colors(present_svtypes):
    return {
        sv: SVTYPE_COLORS.get(sv, SVTYPE_COLORS["UNKNOWN"])
        for sv in present_svtypes
    }


def to_int(value, default=None):
    try:
        if value in {"", ".", "NA", None}:
            return default
        return int(float(value))
    except Exception:
        return default


def chrom_clean(value):
    chrom = str(value or "").strip()
    if chrom.lower().startswith("chr"):
        chrom = chrom[3:]
    return chrom.upper()


def event_size(row):
    chrom1 = chrom_clean(row.get("chrom1"))
    chrom2 = chrom_clean(row.get("chrom2"))
    if chrom1 and chrom2 and chrom1 != chrom2:
        return None
    start = to_int(row.get("start"))
    end = to_int(row.get("end"))
    pos1 = to_int(row.get("pos1"))
    pos2 = to_int(row.get("pos2"))
    if start is not None and end is not None:
        return abs(end - start) + 1
    if pos1 is not None and pos2 is not None:
        return abs(pos2 - pos1) + 1
    return None


def size_bin_label(size):
    if size is None:
        return None
    for label, low, high in SIZE_BINS:
        if low <= size < high:
            return label
    if size < SIZE_BINS[0][1]:
        return SIZE_BINS[0][0]
    return None


def row_tools(row):
    listed = [x.strip().lower() for x in str(row.get("tools", "")).split(",") if x.strip()]
    tools = [tool for tool in TOOLS if tool in listed]
    if tools:
        return tuple(tools)
    tools = []
    for tool in TOOLS:
        if str(row.get(tool, "0")).strip() == "1":
            tools.append(tool)
    return tuple(tools)


def y_positions():
    return {tool: len(PLOT_TOOLS) - 1 - idx for idx, tool in enumerate(PLOT_TOOLS)}


def summarize(rows, max_intersections):
    combo_counts = Counter()
    combo_svtype_counts = defaultdict(Counter)
    combo_size_svtype_counts = defaultdict(lambda: defaultdict(Counter))
    tool_svtype_counts = {tool: Counter() for tool in TOOLS}
    svtype_total = Counter()
    size_total = Counter()

    for row in rows:
        tools = row_tools(row)
        if not tools:
            continue
        combo = tuple(tool for tool in TOOLS if tool in tools)
        svtype = norm_svtype(row.get("svtype"))
        size_label = size_bin_label(event_size(row))

        combo_counts[combo] += 1
        combo_svtype_counts[combo][svtype] += 1
        svtype_total[svtype] += 1

        if size_label:
            combo_size_svtype_counts[combo][size_label][svtype] += 1
            size_total[size_label] += 1

        for tool in combo:
            tool_svtype_counts[tool][svtype] += 1

    sorted_combos = sorted(
        combo_counts.items(),
        key=lambda x: (-x[1], -len(x[0]), ",".join(x[0])),
    )[:max_intersections]
    return sorted_combos, combo_svtype_counts, combo_size_svtype_counts, tool_svtype_counts, svtype_total, size_total


def write_tables(outdir, pair_id, rows, sorted_combos, combo_svtype_counts, combo_size_svtype_counts, svtype_total, size_total):
    present_svtypes = ordered_svtypes(svtype_total)
    summary_path = outdir / "summary" / f"{pair_id}.upset_v7_svtype_size.summary.tsv"
    tumor_id = rows[0].get("tumor_id", "") if rows else ""
    normal_id = rows[0].get("normal_id", "") if rows else ""
    with open(summary_path, "wt") as out:
        out.write("pair_id\ttumor_id\tnormal_id\tmerged_sv_clusters\tshown_intersections\n")
        out.write(f"{pair_id}\t{tumor_id}\t{normal_id}\t{len(rows)}\t{len(sorted_combos)}\n")

    counts_path = outdir / "tables" / f"{pair_id}.upset_v7_svtype_size.counts.tsv"
    with open(counts_path, "wt") as out:
        out.write("pair_id\tcategory\tintersection_rank\tcombination\tname\tsvtype\tcount\n")
        for sv in present_svtypes:
            out.write(f"{pair_id}\tsvtype_total\tNA\tNA\tNA\t{sv}\t{svtype_total.get(sv, 0)}\n")
        for label, _low, _high in SIZE_BINS:
            out.write(f"{pair_id}\tsize_total\tNA\tNA\t{label}\tNA\t{size_total.get(label, 0)}\n")
        for rank, (combo, value) in enumerate(sorted_combos, start=1):
            combo_label = ",".join(combo)
            out.write(f"{pair_id}\tintersection_total\t{rank}\t{combo_label}\tNA\tNA\t{value}\n")
            for sv in present_svtypes:
                out.write(f"{pair_id}\tintersection_svtype\t{rank}\t{combo_label}\tNA\t{sv}\t{combo_svtype_counts[combo].get(sv, 0)}\n")
            for size_label, _low, _high in SIZE_BINS:
                for sv in present_svtypes:
                    out.write(
                        f"{pair_id}\tintersection_size_svtype\t{rank}\t{combo_label}\t{size_label}\t{sv}\t"
                        f"{combo_size_svtype_counts[combo][size_label].get(sv, 0)}\n"
                    )


def draw_svtype_legend(ax_empty, present_svtypes, colors):
    from matplotlib.patches import Patch

    if not present_svtypes:
        return

    handles = [Patch(facecolor=colors[sv], edgecolor="none", label=sv) for sv in present_svtypes]
    ax_empty.legend(
        handles=handles,
        loc="center left",
        bbox_to_anchor=(0.50, 0.63),
        frameon=False,
        ncol=1,
        fontsize=13,
        handlelength=2.2,
        handleheight=1.25,
        handletextpad=0.9,
        labelspacing=1.0,
        borderaxespad=0,
    )


def draw_top_stacked_intersections(ax_bar, sorted_combos, combo_svtype_counts, present_svtypes, colors):
    totals = [value for _combo, value in sorted_combos]
    x = list(range(len(sorted_combos)))
    bottom = [0] * len(sorted_combos)
    for sv in reversed(present_svtypes):
        vals = [combo_svtype_counts[combo].get(sv, 0) for combo, _value in sorted_combos]
        ax_bar.bar(x, vals, bottom=bottom, color=colors[sv], width=0.56)
        bottom = [a + b for a, b in zip(bottom, vals)]
    max_total = max(totals) if totals else 1
    ax_bar.set_ylim(0, max_total * 1.20 + 1)
    ax_bar.set_ylabel("Intersection size", fontsize=12)
    ax_bar.set_xticks([])
    ax_bar.spines[["right", "top"]].set_visible(False)
    ax_bar.spines["left"].set_linewidth(1.0)
    ax_bar.grid(axis="y", color="#b7b7b7", lw=0.9)
    ax_bar.set_axisbelow(True)
    for xi, total in zip(x, totals):
        ax_bar.text(xi, total + max_total * 0.015, str(total), ha="center", va="bottom", fontsize=10)


def draw_upset_matrix(ax_labels, ax_matrix, sorted_combos):

    navy = "#407A7F"

    inactive = "#DCEBF6"

    row_band = "#F4F9FD"
    yp = y_positions()
    x = list(range(len(sorted_combos)))

    for idx, tool in enumerate(PLOT_TOOLS):
        y = yp[tool]
        if idx % 2 == 1:
            ax_labels.axhspan(y - 0.40, y + 0.40, color=row_band, zorder=0)
            ax_matrix.axhspan(y - 0.40, y + 0.40, color=row_band, zorder=0)

    for tool in PLOT_TOOLS:
        ax_matrix.scatter(x, [yp[tool]] * len(x), s=210, color=inactive, edgecolors="none", zorder=1)
    for xi, (combo, _value) in enumerate(sorted_combos):
        ys = [yp[t] for t in combo if t in yp]
        ax_matrix.scatter([xi] * len(ys), ys, s=230, color=navy, edgecolors="none", zorder=3)
        if len(ys) > 1:
            ax_matrix.plot([xi, xi], [min(ys), max(ys)], color=navy, lw=2.4, solid_capstyle="round", zorder=2)

    ax_matrix.set_xlim(-0.7, max(0, len(sorted_combos) - 0.3))
    ax_matrix.set_ylim(-0.7, len(PLOT_TOOLS) - 0.3)
    ax_matrix.set_yticks([])
    ax_matrix.set_xticks([])
    ax_matrix.spines[["right", "top", "left"]].set_visible(False)
    ax_matrix.spines["bottom"].set_linewidth(1.0)

    ax_labels.set_xlim(0, 1)
    ax_labels.set_ylim(-0.7, len(PLOT_TOOLS) - 0.3)
    ax_labels.axis("off")
    label_texts = []
    for tool in PLOT_TOOLS:
        label_texts.append(
            ax_labels.text(0.97, yp[tool], TOOL_LABELS[tool], ha="right", va="center", fontsize=12)
        )
    return label_texts


def draw_left_stacked_tool_bars(ax_set, tool_svtype_counts, present_svtypes, colors):
    row_band = "#F4F9FD"
    yp = y_positions()
    y_ticks = [yp[t] for t in PLOT_TOOLS]
    totals = {tool: sum(tool_svtype_counts.get(tool, Counter()).values()) for tool in PLOT_TOOLS}
    max_total = max(totals.values()) if totals else 1

    for idx, tool in enumerate(PLOT_TOOLS):
        y = yp[tool]
        if idx % 2 == 1:
            ax_set.axhspan(y - 0.40, y + 0.40, color=row_band, zorder=0)

    bottoms = {tool: 0 for tool in PLOT_TOOLS}
    for sv in reversed(present_svtypes):
        values = [tool_svtype_counts.get(tool, Counter()).get(sv, 0) for tool in PLOT_TOOLS]
        lefts = [bottoms[tool] for tool in PLOT_TOOLS]
        ax_set.barh(y_ticks, values, left=lefts, color=colors[sv], height=0.56)
        for tool, value in zip(PLOT_TOOLS, values):
            bottoms[tool] += value

    ax_set.set_ylim(-0.7, len(PLOT_TOOLS) - 0.3)
    ax_set.set_yticks([])
    ax_set.set_xlim(max_total * 1.45 + 1, 0)
    ax_set.spines[["right", "top", "left"]].set_visible(False)
    ax_set.spines["bottom"].set_linewidth(1.0)
    ax_set.grid(axis="x", color="#dddddd", lw=0.7)
    ax_set.set_axisbelow(True)
    for y, tool in zip(y_ticks, PLOT_TOOLS):
        value = totals[tool]
        ax_set.text(value + max_total * 0.03 + 0.5, y, str(value), va="center", ha="right", fontsize=9, clip_on=False, zorder=10)


def restore_v7_left_bar_position(fig, ax_set, label_texts):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    label_left = min(
        fig.transFigure.inverted().transform_bbox(text.get_window_extent(renderer)).x0
        for text in label_texts
    )
    set_pos = ax_set.get_position()
    target_right = label_left - 0.008
    if target_right > set_pos.x1:
        ax_set.set_position([target_right - set_pos.width, set_pos.y0, set_pos.width, set_pos.height])


def draw_downward_size_track(ax_size, sorted_combos, combo_size_svtype_counts, present_svtypes, colors):
    y_step = 1.55
    size_labels = [label for label, _low, _high in SIZE_BINS]
    y_map = {label: idx * y_step for idx, label in enumerate(size_labels)}
    # Small horizontal jitter within each UpSet column.
    # Keep all SVTYPE points inside their own column and avoid overlap.
    max_jitter = 0.30
    if len(present_svtypes) == 1:
        sv_offsets = {present_svtypes[0]: 0.0}
    else:
        positions = [
            -max_jitter + i * (2 * max_jitter / (len(present_svtypes) - 1))
            for i in range(len(present_svtypes))
        ]
        sv_offsets = {
            sv: offset
            for sv, offset in zip(present_svtypes, positions)
        }

    for xi, (combo, _value) in enumerate(sorted_combos):
        for size_label in size_labels:
            sv_counter = combo_size_svtype_counts[combo][size_label]
            for sv in present_svtypes:
                count = sv_counter.get(sv, 0)
                if count <= 0:
                    continue
                point_size = 26 + min(230, count ** 0.50 * 48)
                ax_size.scatter(
                    xi + sv_offsets.get(sv, 0.0),
                    y_map[size_label],
                    s=point_size,
                    color=colors[sv],
                    alpha=0.88,
                    edgecolors="none",
                    linewidth=0,
                    zorder=3,
                )

    ax_size.set_xlim(-0.5, max(0, len(sorted_combos) - 1 + 0.5))
    ax_size.set_ylim(-0.95, (len(size_labels) - 1) * y_step + 0.95)
    ax_size.invert_yaxis()
    ax_size.set_yticks([y_map[label] for label in size_labels])
    ax_size.set_yticklabels(size_labels, fontsize=9)
    ax_size.set_xticks([])
    ax_size.set_ylabel("SV length", fontsize=10)
    ax_size.set_xlabel("Length distribution aligned with UpSet columns", fontsize=10)
    # Vertical separators: same x centers as UpSet columns.
    for xi in range(len(sorted_combos) - 1):
        ax_size.axvline(
            xi + 0.5,
            color="#D0D0D0",
            linestyle="--",
            linewidth=0.7,
            zorder=0,
        )
    ax_size.grid(axis="y", color="#dddddd", lw=0.7)
    ax_size.spines[["right", "top"]].set_visible(False)
    ax_size.spines["bottom"].set_linewidth(1.0)


def plot_pair(outdir, pair_id, rows, args):
    plt = import_matplotlib()
    (
        sorted_combos,
        combo_svtype_counts,
        combo_size_svtype_counts,
        tool_svtype_counts,
        svtype_total,
        size_total,
    ) = summarize(rows, args.max_intersections)
    write_tables(outdir, pair_id, rows, sorted_combos, combo_svtype_counts, combo_size_svtype_counts, svtype_total, size_total)
    present_svtypes = ordered_svtypes(svtype_total)
    colors = svtype_colors(present_svtypes)

    patient = pair_id.replace("_vs_", "_")
    if rows:
        tumor = rows[0].get("tumor_id", "")
        normal = rows[0].get("normal_id", "")
        if tumor and normal:
            if tumor.endswith("T"):
                patient = tumor[:-1]
            else:
                patient = tumor

    fig_width = max(14.5, min(24, 8.2 + 0.58 * max(1, len(sorted_combos))))
    fig = plt.figure(figsize=(fig_width, 11.7), facecolor="white")
    gs = fig.add_gridspec(
        3,
        3,
        width_ratios=[1.28, 1.95, 9.0],
        height_ratios=[3.65, 2.55, 2.85],
        wspace=0.02,
        hspace=0.06,
    )
    ax_empty = fig.add_subplot(gs[0, :2])
    ax_bar = fig.add_subplot(gs[0, 2])
    ax_set = fig.add_subplot(gs[1, 0])
    ax_labels = fig.add_subplot(gs[1, 1])
    ax_matrix = fig.add_subplot(gs[1, 2], sharex=ax_bar)
    ax_empty2 = fig.add_subplot(gs[2, :2])
    ax_size = fig.add_subplot(gs[2, 2], sharex=ax_bar)

    ax_set.set_zorder(4)
    ax_labels.set_zorder(3)
    ax_set.patch.set_alpha(0)
    ax_empty.axis("off")
    ax_empty2.axis("off")
    fig.suptitle(f"SV Callers Concordance - Patient: {patient}", fontsize=19, y=0.985)

    if sorted_combos:
        draw_svtype_legend(ax_empty, present_svtypes, colors)
        draw_top_stacked_intersections(ax_bar, sorted_combos, combo_svtype_counts, present_svtypes, colors)
        label_texts = draw_upset_matrix(ax_labels, ax_matrix, sorted_combos)
        draw_left_stacked_tool_bars(ax_set, tool_svtype_counts, present_svtypes, colors)
        draw_downward_size_track(ax_size, sorted_combos, combo_size_svtype_counts, present_svtypes, colors)
    else:
        ax_bar.text(0.5, 0.5, "No SV events", ha="center", va="center", fontsize=13)
        for ax in [ax_bar, ax_set, ax_labels, ax_matrix, ax_size]:
            ax.axis("off")
        label_texts = []

    fig.subplots_adjust(left=0.085, right=0.985, bottom=0.07, top=0.90)
    if label_texts:
        restore_v7_left_bar_position(fig, ax_set, label_texts)
    png = outdir / "plots" / f"{pair_id}.upset_v7_svtype_size.png"
    fig.savefig(png, dpi=args.png_dpi, bbox_inches="tight")
    plt.close(fig)
    return png


def combine_summaries(outdir):
    files = sorted((outdir / "summary").glob("*.upset_v7_svtype_size.summary.tsv"))
    combined = outdir / "all_pairs.upset_v7_svtype_size.summary.tsv"
    wrote_header = False
    with open(combined, "wt") as out:
        for path in files:
            with open(path, "rt") as handle:
                header = handle.readline()
                body = handle.read()
            if not wrote_header:
                out.write(header)
                wrote_header = True
            if body:
                out.write(body)
    return combined


def main():
    args = parse_args()
    outdir = Path(args.outdir)
    mkdirs(outdir)
    paths = event_paths(args)
    if not paths:
        raise SystemExit(f"No event tables found in {args.events_dir}")

    for path in paths:
        if not path.exists():
            raise SystemExit(f"Event table not found: {path}")
        rows = read_events(path)
        pair_id = rows[0].get("pair_id") if rows else path.name.replace(".merged_sv_events.tsv", "")
        png = plot_pair(outdir, pair_id, rows, args)
        print(f"[DONE] {pair_id}: {png}")

    combined = combine_summaries(outdir)
    print(f"[SUMMARY] {combined}")


if __name__ == "__main__":
    main()
()


### run_upset.sh：

In [ ]:
#!/bin/bash
#SBATCH --job-name=upset_v7_sv
#SBATCH --nodes=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=4G
#SBATCH --output=/mnt/home/ygjx/chenkejin/80_upset/slurm_logs/upset_v7_%A_%a.out
#SBATCH --error=/mnt/home/ygjx/chenkejin/80_upset/slurm_logs/upset_v7_%A_%a.err

set -euo pipefail

source /mnt/home/ygjx/chenkejin/anaconda3/etc/profile.d/conda.sh
conda activate wgs_bam_qc

SCRIPT="/mnt/home/ygjx/chenkejin/80_upset/scripts/plot_upset_v7_svtype_size_from_events.py"
MANIFEST="/mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv"
EVENTS_DIR="/mnt/home/ygjx/chenkejin/80_upset/events"
OUTDIR="/mnt/home/ygjx/chenkejin/80_upset/upset_v7_svtype_size"

mkdir -p "${OUTDIR}" /mnt/home/ygjx/chenkejin/80_upset/slurm_logs

python "${SCRIPT}" \
  --task-id "${SLURM_ARRAY_TASK_ID}" \
  --manifest "${MANIFEST}" \
  --events-dir "${EVENTS_DIR}" \
  --outdir "${OUTDIR}" \
  --max-intersections 12 \
  --png-dpi 220


## 二、步骤

### 提取画图所用信息：

In [ ]:
mkdir -p /mnt/home/ygjx/chenkejin/80_upset/events

python build_sv_upset_events_from_reclass_vcfs.py \
  --result-root /mnt/home/ygjx/chenkejin/share_group_folder_ygjx/Pancreatic_datasets/PDAC_WGS/sv_tools_results/sv_tools_results_bnd_reclass \
  --outdir /mnt/home/ygjx/chenkejin/80_upset \
  --pair-list /mnt/home/ygjx/chenkejin/80_upset/80_pairs.normal_tumor.tsv \
  --max-distance 1000 \
  --same-type-only

### 画图：

In [ ]:
N=$(($(wc -l < /mnt/home/ygjx/chenkejin/80_upset/80_pairs.six_sv_tools.manifest.new.tsv) - 1))

sbatch --array=1-${N}%20 \
  /mnt/home/ygjx/chenkejin/80_upset/scripts/run_upset.sh